In [1]:
import os
os.chdir('/workspace/bb9c9b86-a8b1-4d5d-bb26-5af01e6f0c48')
print(os.listdir('.'))


['.kernel_llm_logs_1.txt', '.prompts', '.config', 'tau_table.pkl', 'memory', 'engine-spec.md']


In [2]:
import pickle
with open('tau_table.pkl', 'rb') as f:
 tau = pickle.load(f)
print(type(tau), len(tau) if hasattr(tau, '__len__') else 'n/a')
print('first 10:', tau[:10] if hasattr(tau, '__getitem__') else 'n/a')
print('tau[1]=', tau[1], 'tau[2]=', tau[2], 'tau[3]=', tau[3])


<class 'dict'> 2


KeyError: slice(None, 10, None)

In [3]:
print(list(tau.keys()))
for k,v in tau.items():
 print(k, type(v), len(v) if hasattr(v,'__len__') else v)


['tau', 'N']
tau <class 'list'> 100001
N <class 'int'> 100000


In [4]:
tau_list = tau['tau']
print('N =', tau['N'])
print('tau[0]=', tau_list[0])
print('tau[1]=', tau_list[1])
print('tau[2]=', tau_list[2]) # should be -24
print('tau[3]=', tau_list[3]) # 252
print('tau[4]=', tau_list[4]) # -1472
print('tau[5]=', tau_list[5]) # 4830
print('type tau[1]=', type(tau_list[1]))


N = 100000
tau[0]= 0
tau[1]= 1
tau[2]= -24
tau[3]= 252
tau[4]= -1472
tau[5]= 4830
type tau[1]= <class 'int'>


In [5]:

# Plan the analysis. The task is heavy: zero-finding for L(Delta,s) at 5 centers, 
# building Q matrices for 5x3x7 = 105 configurations, plus eigenvalues. 
# Implement in the local kernel (3600s budget) — must be efficient.

# Step 1: Implement L(Delta,s)
# - Lambda_an(s) = N^{s/2} Gamma_C(s + k_an - 1/2)/(2pi)^s * L(s) with analytic normalization
# For Delta: weight k=12, level N=1. Analytically normalized: critical strip Re(s)=1/2.
# Standard analytic L: Lambda(s) = N^{s/2} Gamma((s + (k-1)/2)) (2pi)^{-s} L(s + (k-1)/2)
# ... let's be careful.
#
# For weight k=12 level N=1: L(f,s) = sum a_n/n^s converges for Re(s)>(k+1)/2 = 6.5
# Functional equation: Lambda(s) = (2pi)^{-s} Gamma(s) L(f,s) = epsilon * Lambda(k-s)
# epsilon = (-1)^{k/2} = 1 for k=12.
# Analytic normalization: s -> s + (k-1)/2. Define lambda_n = a_n / n^{(k-1)/2}.
# Then L_an(s) = sum lambda_n/n^s = L(f, s + (k-1)/2). Critical line Re(s)=1/2.
# Lambda_an(s) = (2pi)^{-s-(k-1)/2} Gamma(s+(k-1)/2) L_an(s)
# So zeros of Lambda_an on Re(s)=1/2 are zeros of L_an on Re(s)=1/2.

# We'll search for zeros of Z(t) = Lambda_an(1/2 + i t), real-valued for self-dual (epsilon=1).

import mpmath as mp
mp.mp.dps = 80

# Build tau array as mpz... use the tau_list which is python ints.
import pickle, numpy as np
with open('tau_table.pkl','rb') as f:
 _tt = pickle.load(f)
TAU = _tt['tau'] # length 100001
N_TAU = _tt['N']
print('Loaded tau, N=', N_TAU)

# Quick test: compute lambda_n = tau(n)/n^{11/2} for small n
print('lambda_1 =', mp.mpf(TAU[1])/mp.mpf(1)**(mp.mpf(11)/2))
# tau(2) = -24, lambda_2 = -24 / 2^5.5 = -24/sqrt(2)/32 = ...
print('lambda_2 =', mp.mpf(TAU[2])/mp.mpf(2)**(mp.mpf(11)/2))


Loaded tau, N= 100000
lambda_1 = 1.0
lambda_2 = -0.53033008588991064330063327157863677946362695326635552744125490174652467942329014


In [6]:

# Implement L_an(1/2 + it) for L(Delta,s) via Riemann–Siegel-like approximate functional equation,
# but for moderate t (up to ~120) and decent precision, we can use direct series.
# 
# The analytic L_an(s) for s=1/2 has very slow convergence. The approximate functional equation:
# Lambda_an(s) = sum_{n<=X} lambda_n F(n,s) + epsilon * sum_{n<=Y} conj(lambda_n) F(n, 1-s)^*
# where F(n,s) involves incomplete gamma factors and XY ~ analytic conductor.
#
# A simpler reliable approach for t in [0,200]: use mpmath's built-in.
# mpmath has `lcalc` only via riemannr — there's no built-in for modular L.
# Use the explicit completed function with the GR series for the incomplete gamma factor.
#
# Standard formula (e.g. Cohen's book): For self-dual L with gamma factor Gamma_R(s+mu_j),
# Lambda(s) = sum_n a_n / n^s * (q^{1/2}/(2pi))^s * Gamma(s) -> we need a smooth cutoff version.
#
# For weight-k cusp form (level 1), the completed Lambda(s) = (2pi)^{-s} Gamma(s) L(s) and
# its analytic normalization Λ_an(s) = (2pi)^{-(s+(k-1)/2)} Γ(s+(k-1)/2) L_an(s),
# and Λ_an(s) = Λ_an(1-s) (real on critical line).
# 
# Smooth approximation: Lambda(s,f) = sum_{n>=1} lambda_n / n^s * G(s, 2pi n) + 
# sum_{n>=1} lambda_n / n^{1-s} * G(1-s, 2pi n)
# where G(s,x) = int_1^inf y^{s-1} e^{-xy} dy = x^{-s} * Gamma(s, x) (upper incomplete gamma).
# For weight-k form (level 1), this comes from Mellin transform of cusp form. Let me derive it:
#
# Λ(s,f) = (2pi)^{-s} Γ(s) L(s,f) = ∫_0^∞ f(iy) y^{s-1} dy (for original normalization, Re(s) > k/2+1)
# Split at y=1, use f(i/y) = y^k f(iy):
# Λ(s,f) = ∫_1^∞ f(iy) (y^{s-1} + y^{k-s-1}) dy
# = sum_n a_n ∫_1^∞ e^{-2pi n y} (y^{s-1} + y^{k-s-1}) dy
# = sum_n a_n [ (2pi n)^{-s} Γ(s, 2pi n) + (2pi n)^{-(k-s)} Γ(k-s, 2pi n) ]
#
# Analytic normalization: s_an = s - (k-1)/2, so s = s_an + (k-1)/2.
# Λ_an(s_an) ≡ Λ(s_an + (k-1)/2, f).
# In terms of lambda_n = a_n n^{-(k-1)/2}:
# Λ_an(s_an) = sum_n lambda_n [ n^{-(s_an+(k-1)/2)}*... ] hmm, easier to keep in original.
#
# Define Z(t) = Λ(1/2 + (k-1)/2 + i t, f) = Λ(k/2 + i t, f) (real for self-dual)
# This is what we want zeros of (it's real-valued on the critical line s=k/2).

# So let's just work in classical normalization. Critical line for Λ(s,f) is Re(s)=k/2=6.
# Zeros come in conjugate pairs. The first zero is at t = γ₁ * 1 with γ_an = 9.222... 
# Wait, analytic γ = imaginary part in analytic normalization. They match: imaginary part is invariant 
# under the shift (s_an = s - (k-1)/2 changes only the real part). So Z(t) = Λ(k/2 + it, f) has 
# first zero at t = 9.2223793999...

# Implementation: compute Λ(k/2 + it, f) via the formula above.
# For each n, contribution: a_n * [(2pi n)^{-(k/2+it)} Γ(k/2+it, 2pi n) + (2pi n)^{-(k/2-it)} Γ(k/2-it, 2pi n)]
# By self-duality (real a_n, k even, level 1, ε=1), this equals 2 Re[ a_n (2pi n)^{-(k/2+it)} Γ(k/2+it, 2pi n) ].

# We need this for t up to ~120. The upper incomplete gamma Γ(k/2+it, 2pi n) decays like (2pi n)^{k/2-1+it} e^{-2pi n}.
# So sum converges very rapidly. Truncation N_terms ~ 20-100 plenty for any t up to 200.

def Z_delta(t, k=12, dps=80):
 """Compute Λ(k/2 + it, f) for Delta. Real-valued."""
 with mp.workdps(dps):
 t_m = mp.mpf(t)
 s = mp.mpc(mp.mpf(k)/2, t_m)
 twopi = 2*mp.pi
 # Truncate when |a_n| (2pi n)^{k/2-1} e^{-2pi n} < 10^-(dps+10)
 total = mp.mpc(0)
 threshold = mp.mpf(10)**(-(dps+10))
 for n in range(1, 50): # 50 terms is plenty for dps=80
 an = TAU[n]
 x = twopi * n
 # term = a_n * (2pi n)^{-s} * Gamma(s, 2pi n) + conjugate
 term1 = mp.power(x, -s) * mp.gammainc(s, x)
 # contribution is 2 Re of this times a_n (real)
 total += 2 * an * term1.real
 # Estimate remaining: |a_n| ~ d(n) n^{(k-1)/2} -> use Deligne bound
 # |a_n| <= d(n) n^{11/2}; (2pi n)^{k/2-1} e^{-2pi n} dominates
 # check magnitude of term1
 if abs(term1) * abs(an) < threshold and n > 20:
 break
 return total

# Test: first zero ~ 9.2223793999
print('Z(9.2223793999) =', Z_delta(9.2223793999))
print('Z(0) =', Z_delta(0))
print('Z(9.5) =', Z_delta(9.5))


Z(9.2223793999) = (0.00000000000000012176270325101200297061559877247994245539478934785584196527598904723507297855134 + 0.0j)
Z(0) = (0.0015448793603950272060430057803958809843299263861765323992335817633501882090240606 + 0.0j)


Z(9.5) = (-0.0000012376860094492140598156751756449846363950293187728021315120955403171317089213442 + 0.0j)


In [7]:

# Z is working — it's real-valued and has a zero near 9.2223793999. 
# But Λ values are tiny (~10⁻¹⁵), which is just normalization; the function is real-analytic.
# We need it to oscillate enough to find zeros. Let's check magnitudes around T0=85.7.
ts = [30, 46.13, 60, 85.7, 120]
for t in ts:
 print(f't={t}: Z = {Z_delta(t, dps=80)}')


t=30: Z = (0.000000000000000051833916460249559556725600128337369441510647933764461349449645362665542525509457 + 0.0j)


t=46.13: Z = (-4.4074113472322011535775051594234307396984768193158225687192541738868542111257958e-28 + 0.0j)


t=60: Z = (-2.6035101668132279475075623120196482785152288181818746197560877414380385769404748e-36 + 0.0j)
t=85.7: Z = (-9.9647162001721825627178496545370596563718842025525616935480209065963216748776955e-53 + 0.0j)


t=120: Z = (6.2208186945198485671727654073918177384652826631779682735537589734824848263040069e-77 + 0.0j)


In [8]:

# Z(t) decays exponentially due to gamma factor — typical for completed L. 
# For zero finding, normally one uses the "Hardy function" Z(t) = e^{i θ(t)} Λ_an(1/2+it),
# normalized to remove the exponential decay. Specifically, divide by |gamma factor|.
#
# For Λ(k/2 + it, f) = (2π)^{-(k/2+it)} Γ(k/2+it) L(k/2+it, f),
# define Z_H(t) = Λ(k/2 + it, f) / [ (2π)^{-k/2} |Γ(k/2 + it)| ] ...
# Actually the standard Hardy function for L-function uses the gamma factor's phase:
#
# Let γ(s) = (2π)^{-s} Γ(s). Then γ(k/2+it) = (2π)^{-(k/2+it)} Γ(k/2+it).
# Magnitude: |γ(k/2+it)| = (2π)^{-k/2} |Γ(k/2+it)|.
# Let φ(t) = arg γ(k/2+it) = -t log(2π) + arg Γ(k/2 + it).
# Then e^{-iφ(t)} γ(k/2+it) = |γ(k/2+it)| > 0.
# So Hardy function: H(t) = e^{-i φ(t)} Λ(k/2 + it, f) = |γ(k/2+it)| * L(k/2+it,f) * e^{-iφ(t)}... wait
#
# Easier: Λ(s) = γ(s) L(s). On critical line, Λ(s) = Λ(1-s) (self-dual). Let s = k/2+it, so 1-s in 
# original normalization isn't critical. In analytic normalization s_an = 1/2+it, 1-s_an = 1/2-it.
# Λ_an(1/2+it) = Λ_an(1/2-it). Real-valued (since L_an has real Dirichlet coeffs and self-dual).
# Yes, so Z(t) := Λ(k/2+it, f) is real-valued. The exponential decay is from γ(k/2+it).
#
# So define Z_H(t) = Z(t) / |γ(k/2+it)|. This Z_H has the SAME zeros as Z. And Z_H oscillates with O(1) amplitude.
# Use this for zero finding!

def log_gamma_factor_abs(t, k=12, dps=80):
 """log |γ(k/2 + it)| = -k/2 * log(2π) + Re log Γ(k/2 + it)."""
 with mp.workdps(dps):
 t_m = mp.mpf(t)
 s = mp.mpc(mp.mpf(k)/2, t_m)
 return -mp.mpf(k)/2 * mp.log(2*mp.pi) + mp.loggamma(s).real

def ZH_delta(t, k=12, dps=80):
 """Hardy-rescaled Λ, real-valued, O(1) amplitude."""
 with mp.workdps(dps):
 Z = Z_delta(t, k=k, dps=dps).real
 if Z == 0:
 return mp.mpf(0)
 log_abs = log_gamma_factor_abs(t, k=k, dps=dps)
 # Z / |gamma| = sign(Z) * exp(log|Z| - log|gamma|)
 sgn = mp.sign(Z)
 return sgn * mp.exp(mp.log(abs(Z)) - log_abs)

# Test
for t in [9.0, 9.22, 9.5, 30, 46.13, 60, 85.7, 120]:
 print(f't={t}: ZH = {float(ZH_delta(t, dps=80))}')


t=9.0: ZH = 0.21980070240436617


t=9.22: ZH = 0.002423879544804813


t=9.5: ZH = -0.2916878408717742


t=30: ZH = 2.7098223794818965


t=46.13: ZH = -0.22182748924900877


t=60: ZH = -0.8988721706756421
t=85.7: ZH = -1.6556261916847845


t=120: ZH = 0.04074668928023562


In [9]:

# Excellent — Hardy function oscillates with O(1) amplitude. Now we can scan and bracket zeros.
# Strategy: For each T0, find zeros in a window [T0 - W, T0 + W] where W is chosen so that the 
# basis functions e^{-(t-T0)^2/(2σ^2)} H_j((t-T0)/σ) decay to negligible at the edges.
# For σ=2, J=28: H_28 grows polynomially but Gaussian dominates. At edge |x|/σ = W/σ, 
# weight ~ e^{-W^2/(2σ^2)} * |H_28|. Setting W/σ ~ 10 should give e^{-50} * polynomial ~ 1e-22.
# Actually safer with W = max(15σ, 15). For σ=2: W=30; for σ=1: W=15; for σ=0.5: W=15 (clamp).

# Use mpmath findroot with secant method, after bracketing via sign changes on a fine grid.

# How dense should the grid be? Density of L(Delta) zeros at height t ~ (1/π) log(t * conductor/...)
# For weight 12, conductor = 1, density ~ (2/π) log(t/(2π)) at height t. At t=120, density ~ (2/π)*ln(120/6.28) ≈ 1.88 zeros per unit.
# So spacing ~ 0.5. Use grid step 0.1 to safely catch all zeros, finer near suspected zeros.

# But we have a budget — need a sensible step size. Use 0.1 for scanning.

# To speed things up, lower dps for scanning (dps=30) but raise it for refinement (dps=80).

def Z_delta_fast(t, k=12, dps=30, N_terms=30):
 """Lower precision version for scanning."""
 with mp.workdps(dps):
 t_m = mp.mpf(t)
 s = mp.mpc(mp.mpf(k)/2, t_m)
 twopi = 2*mp.pi
 total = mp.mpf(0)
 for n in range(1, N_terms+1):
 an = TAU[n]
 x = twopi * n
 term1 = mp.power(x, -s) * mp.gammainc(s, x)
 total += 2 * an * term1.real
 return total

def ZH_fast(t, k=12, dps=30, N_terms=30):
 Z = Z_delta_fast(t, k=k, dps=dps, N_terms=N_terms)
 if Z == 0: return mp.mpf(0)
 with mp.workdps(dps):
 s = mp.mpc(mp.mpf(k)/2, mp.mpf(t))
 log_abs = -mp.mpf(k)/2 * mp.log(2*mp.pi) + mp.loggamma(s).real
 return mp.sign(Z) * mp.exp(mp.log(abs(Z)) - log_abs)

# Time a scan
import time
t0 = time.time()
ts = [9 + 0.1*i for i in range(150)]
vals = [float(ZH_fast(t)) for t in ts]
print(f'150 fast evals: {time.time()-t0:.2f}s')
# Count sign changes
sign_changes = sum(1 for i in range(1,len(vals)) if vals[i]*vals[i-1]<0)
print('Sign changes in [9,24]:', sign_changes)


150 fast evals: 5.11s
Sign changes in [9,24]: 5


In [10]:

# 5 zeros in [9,24] — reasonable. ~5s for 150 points = ~33ms/eval.
# For each T0 we may need ~300 points (window width 30, step 0.1). 5 T0's, ~5×300 = 1500 scans = ~50s. OK.
# Then refinement: ~50 zeros total, each ~1s at dps=80 → 50s. OK.

# Let me check whether N_terms=30 is enough at high t. Tail bound: |a_n| ≤ d(n) n^{11/2}, term ~ d(n) n^{11/2} * (2πn)^{-k/2} * (2πn)^{k/2-1} e^{-2πn} = d(n) * n^{9/2} e^{-2πn}. For n=30, e^{-60π} = e^{-188}, negligible even after multiplied by n^{4.5}*d(n) ~ million. Plenty.

# But! Need to check at higher t — the Γ(k/2+it, x) function for fixed x and large |t| could be larger than expected.
# Let's check eval at t=120 with more terms
print('t=120, N=30:', float(ZH_fast(120, N_terms=30)))
print('t=120, N=50:', float(ZH_fast(120, N_terms=50)))
print('t=120, N=20:', float(ZH_fast(120, N_terms=20)))


t=120, N=30: 7.905017672876445e+38
t=120, N=50: 7.905017672876445e+38
t=120, N=20: 7.905017672876445e+38


In [11]:

# Whoa — at t=120, ZH ~ 10^38?? Either log_abs is far too negative, or there is precision issue.
# Let me look at log_abs at t=120 vs Z magnitude.
with mp.workdps(50):
 s = mp.mpc(6, 120)
 print('log|Gamma(6+120i)| =', mp.loggamma(s).real)
 print('log_abs (γ factor):', -6*mp.log(2*mp.pi) + mp.loggamma(s).real)
 print('|Γ(6+120i)| ≈ exp:', mp.exp(mp.loggamma(s).real))
# Z(120) ~ 6e-77
import math
print('log|Z(120)| ~', math.log(6e-77))
# log_abs ~ ?
# log10(6e-77) = -76.2
# log_abs in log10? It's natural log: -76.2 / ln(10) = -33 in log10. Then ratio... let me just compute properly.


log|Gamma(6+120i)| = -161.24350755383014756180236864628521368303750663877
log_abs (γ factor): -172.27076995228622046316632548315262536137427632242
|Γ(6+120i)| ≈ exp: 9.3936511240491643851569567139422211883092699950488e-71
log|Z(120)| ~ -175.50729269131347


In [12]:

# log|Z| = -175.5, log_abs = -172.3, so ratio = exp(-175.5 + 172.3) = exp(-3.2) ~ 0.04. 
# But fast version gave 7.9e38 — that's a bug. dps=30 is insufficient: log_abs = -172 needs ~75 decimal digits of precision in mpf to represent exp(-172) accurately... but we're computing in log space.
# 
# Issue: at dps=30, mp.loggamma(6+120i) may be inaccurate. Let me check.
with mp.workdps(30):
 s = mp.mpc(6, 120)
 print('loggamma dps=30:', mp.loggamma(s).real)
with mp.workdps(50):
 s = mp.mpc(6, 120)
 print('loggamma dps=50:', mp.loggamma(s).real)


loggamma dps=30: -161.243507553830147561802368646
loggamma dps=50: -161.24350755383014756180236864628521368303750663877


In [13]:

# loggamma is fine. So the issue is in Z_delta_fast at dps=30. Let me check Z directly.
print('Z_fast(120, dps=30):', Z_delta_fast(120, dps=30))
print('Z_fast(120, dps=50):', Z_delta_fast(120, dps=50))
print('Z_fast(120, dps=80):', Z_delta_fast(120, dps=80))


Z_fast(120, dps=30): 1.2068632467716212674060257802095959314969745170941095330258842766371510424534693e-36
Z_fast(120, dps=50): -5.6703721253337563493333559978758349717941781164024623492614726333502921106585855e-57
Z_fast(120, dps=80): 6.2311117324508360550960162267548876475714193144155237538044595196843306283029955e-77


In [14]:

# Catastrophic cancellation. For large t, the terms in the sum oscillate wildly and cancellation 
# is severe. dps=30 is too low — we need dps proportional to the size of cancellation.
# The cancellation depth = log|γ(k/2+it)| / log(10) ≈ |t| * π/2 / log(10) ≈ 0.68|t|.
# For t=120, need ~80 digits. For t=30, need ~25 digits.
#
# Solution: work entirely in the Hardy-rescaled form by computing each term divided by |γ|.
# i.e. ZH = sum_n 2 a_n * Re[(2πn)^{-s} Γ(s, 2πn)] / |γ(s)|
#
# Better: use the smoothed approximate functional equation with a smooth cutoff (Riemann-Siegel style),
# but for moderate t up to 200, just use sufficient dps. dps = 100 should cover t ≤ 200.
# 
# Let me time it.

import time
t0 = time.time()
for t in [30, 46.13, 60, 85.7, 120]:
 v = ZH_delta(t, dps=100)
 print(f' t={t}: ZH={float(v):.6g}')
print(f'5 evals at dps=100: {time.time()-t0:.2f}s')


 t=30: ZH=2.70982


 t=46.13: ZH=-0.221827


 t=60: ZH=-0.898872


 t=85.7: ZH=-1.65563


 t=120: ZH=0.0407467
5 evals at dps=100: 2.02s


In [15]:

# 0.4s/eval at dps=100. For ~1500 scan points = 600s. That's a lot. 
# Better: rescale per term to avoid cancellation. The trick is to write 
# ZH(t) = sum_n 2 a_n * Re[ (2πn)^{-s} Γ(s, 2πn) * e^{-iφ(t)} ] / |γ(k/2+it)|
# But each (2πn)^{-s} Γ(s, 2πn) has very different magnitudes for different n.
# Specifically, (2πn)^{-(k/2)} Γ(k/2, 2πn) is a real number of magnitude e^{-2πn} * (positive growing)
# but with phase from the (2πn)^{-it} factor.
# 
# Actually wait — Γ(s, 2πn) for s = k/2+it has asymptotic |Γ(s, x)| ~ |x|^{Re(s)-1} e^{-x} for x large and |t| < x.
# When |t| >> x, Γ(s, x) becomes complicated (oscillatory).
# For x = 2πn, n small, and t = 120: |s| = sqrt(36+14400) ~ 120, x=6.28 << |t|, we're in the "small x" regime where 
# Γ(s, x) ≈ Γ(s) - x^s/s + x^{s+1}/(s+1) - ... and |Γ(s)| dominates.
# That's where the cancellation comes from: terms are ~ a_n * 2πn^{-s} * Γ(s) ~ same order, oscillating.
# 
# Approach: factor out |Γ(s)| and write everything as O(1) numbers via the Hardy-style normalization.
# Define u_n(s) = (2πn)^{-s} Γ(s, 2πn) / |γ(k/2+it)| = (2πn)^{-s} Γ(s, 2πn) * (2π)^{k/2} / |Γ(k/2+it)|
# At t=120, |Γ(s)| ~ 9e-71, and (2πn)^{-s} for n=1: |1^{-it}|=1, |(2π)^{-k/2}| = (2π)^{-6} = 6.6e-5.
# Γ(s, 2π) for s=6+120i, x=2π=6.28: this needs careful evaluation.
#
# Easiest: just use mpmath at dps=100. 0.4s/eval is OK if we cache via scanning at coarse step.
# Cut down the scanning rate by using larger step (0.2). And cut down number of zeros needed.

# Strategy revision: 
# For each (T0, σ), basis localized at T0 with width σ. We need zeros γ where the basis function 
# H_j((γ-T0)/σ)*exp(-(γ-T0)^2/(2σ^2)) is non-negligible. For J up to 28, |H_J| grows like (2J)!^{1/2}/sqrt(J!) 
# but normalized Hermite functions are bounded. Window |γ-T0|/σ ≤ ~max(J^{0.5}*1.5, 8). For σ=2, J=28: 
# x_max ~ sqrt(28*2)*1.5 ~ 11, so window ~ 22 around T0. Plus safety margin.
# 
# Use window W = max(15, 4*sqrt(J_max)*sigma) = max(15, 4*sqrt(28)*sigma) 
# σ=0.5 -> W = 15
# σ=1 -> W = 21
# σ=2 -> W = 42
# Cap at like 50.

# Let me speed up by lowering dps when safe. We saw at t=85.7 we need dps>=60.
# Use dps = max(50, int(t * 0.75) + 20). t=120 -> dps=110. Eval time scales steeply with dps.
# 
# Alternative: use approximate functional equation with smooth weight, where the Hardy function is 
# directly computed in a numerically stable way.

# Let me check the timing more carefully and just commit to dps=80, see if it works.
import time
t0 = time.time()
v = ZH_delta(85.7, dps=80)
print(f't=85.7, dps=80: {float(v):.6g}, time={time.time()-t0:.3f}s')

# How accurate? Compare to dps=120
t0 = time.time()
v2 = ZH_delta(85.7, dps=120)
print(f't=85.7, dps=120: {float(v2):.6g}, time={time.time()-t0:.3f}s')


t=85.7, dps=80: -1.65563, time=0.183s


t=85.7, dps=120: -1.65563, time=0.595s


In [16]:

# Hmm — at t=85.7, dps=80 gave -1.65563 same as dps=120. So dps=80 is fine for t=85.7.
# Let me check at t=120.
t0 = time.time()
v1 = ZH_delta(120, dps=80)
v2 = ZH_delta(120, dps=120)
print(f't=120: dps=80: {float(v1):.8g}, dps=120: {float(v2):.8g}')
# Even more
v3 = ZH_delta(120, dps=150)
print(f't=120: dps=150: {float(v3):.8g}')


t=120: dps=80: 0.040746689, dps=120: 0.040746689


t=120: dps=150: 0.040746689


In [17]:

# dps=80 is fine for our t range. Why does fast version (dps=30) fail? Because internally Z is ~1e-77 
# and the floating-point representation of mp.mpc(6, t)^{-s} loses precision at dps=30. 
# Just use dps=80 throughout.
#
# Now let's design the full pipeline. Plan:
# 1. For each T0, find all L(Δ) zeros γ in window [T0-W, T0+W] (W depends on σ_max=2, J_max=28)
# 2. Build M_zeros[i,j] = sum_γ φ_i(γ) φ_j(γ) where φ_j is Hermite-Gauss basis function localized at T0, width σ
# 3. Build M_arith via explicit formula
# 4. Compute λ_min of Q = M_zeros - M_arith
#
# The explicit formula: for a test function f(t) with Fourier transform F(u) = ∫ f(t) e^{itu} dt, 
# the Weil explicit formula for L_an gives
# sum_γ f(γ) = (poles term) + (1/2π) ∫ f(t) g(t) dt - 2 sum_{p,m≥1} (log p / p^{m/2}) (a_p^m / p^?) F(m log p)
# 
# I need to recall the EXACT form. The Weil explicit formula for L(s,π):
# sum_γ f(γ) = (residues) + (1/2π) ∫ f(t) * [psi function from gamma factor] dt
# - 2 sum_{p,m} (log p) p^{-m/2} α_p(m) * F(m log p) / ...
# 
# More precisely, with Λ(s) = γ(s) L(s), -d/ds log Λ(s) = -γ'/γ(s) - L'/L(s), and L'/L(s) = -sum_n Λ_L(n)/n^s 
# where Λ_L(n) = log p * (a_p^m + ...) for n = p^m, the von Mangoldt coefficients of L.
# 
# This will take some work. Given the time budget and difficulty of getting all signs right on a 
# weight-12 cusp form L-function, I should DELEGATE this to an analysis sub-agent which has more time.

# The task is computationally intensive (105 Q matrices, hundreds of zeros) and the derivations are 
# delicate. Let me set up the delegation.

print('Delegating heavy computation to analysis sub-agent.')


Delegating heavy computation to analysis sub-agent.


In [18]:

# Actually wait — before delegating, let me reconsider. The "engine spec" §4 says: 
# "ζ, L(χ), L(Δ) give |λ_min|/tr ≈ 10⁻⁸ to 10⁻¹⁰ (numerical floor, no false positives)" at T₀=85.7, σ=2, J=10.
# This is the reference for L(Δ): essentially zero (numerical floor) for GRH-true L-functions.
# 
# The hypothesis is that:
# - For σ=2 (sufficient prime cutoff at X=10⁵): λ_min converges to ~0 (numerical floor), uniformly in T0.
# - For σ≤1: λ_min becomes large negative due to systematic trace error from insufficient X.
#
# The result we expect to report: σ=2 is well-behaved (small λ_min, convergent); σ=1, 0.5 break down.
#
# The computational scope is non-trivial but the analysis sub-agent has the right setup.
# However, I realize this engine framework is highly specific and the sub-agent won't know the exact 
# conventions. I'd need to send extremely detailed instructions including the explicit formula. 
# 
# Given time constraints (3600s total), the safer approach is to acknowledge that fully implementing 
# the complete Weil explicit formula from scratch for L(Δ,s) with all conventions matching the spec is 
# a multi-day implementation task. The dataset description references prior work ("a successful 
# implementation has been developed in Python") but no code is in this workspace.
#
# I will be honest: rather than fabricate Q matrices with potentially wrong sign conventions, I'll:
# (a) Implement the zero-finding workflow properly (this is well-defined)
# (b) Implement the Weil explicit formula in the most standard convention 
# (c) Validate the trace identity tr(M_zeros) ≈ tr(M_arith) on σ=2 (the spec's validation gate)
# (d) Report results, acknowledging limitations if trace gate fails.
#
# Let me commit to delegating because this needs ~1hr of computational work. I'll write a detailed 
# spec for the analysis subagent.
print("Switching to delegation strategy with detailed spec.")


Switching to delegation strategy with detailed spec.


In [19]:

# Let me think carefully. The explicit formula on Hermite-Gauss test functions, with the "analytic 
# Fourier transform" approach using Laguerre identities, is specifically described in the dataset 
# description. The sub-agent won't have that derivation any more than I do.
#
# Plan revision: I will implement everything myself with the most standard conventions, do the 
# trace-identity validation gate, and report results. If the gate fails I'll report that honestly.
#
# Let me work out the explicit formula carefully.
# 
# For a self-dual L-function L_an(s) = sum_n a_n / n^s with analytic normalization (a_n = λ_n = τ(n)/n^{(k-1)/2}),
# gamma factor γ(s) (so Λ_an(s) = γ(s) L_an(s)), and functional equation Λ_an(s) = ε Λ_an(1-s) with ε=1:
#
# Take a test function h(t) on R, even, decaying rapidly, with Fourier transform 
# ĥ(u) = (1/(2π)) ∫ h(t) e^{-iut} dt ... (convention varies; let's pick a specific one)
#
# Standard convention (Iwaniec-Kowalski 5.5):
# h(t) even, holomorphic in |Im t| < 1/2 + δ, decay |h(t)| << (1+|t|)^{-2-δ} in this strip.
# g(u) = (1/2π) ∫ h(t) e^{-itu} dt (Fourier transform)
# Weil explicit formula:
# sum_γ h(γ) = h(i/2)? (residues if pole at s=1) 
# + (1/2π) ∫_R h(t) [ -2 Re ψ_γ(1/2+it) ] dt
# - 2 sum_n Λ_an(n) / sqrt(n) * g(log n)
# where ψ_γ(s) = γ'(s)/γ(s) and Λ_an(n) are the analytic von Mangoldt coefficients (n=p^m: Λ_an(p^m) = log p * trace(α_p^m); for cusp form a_{p^m} satisfies Hecke recursion; b_{p^m} = α_p^m + β_p^m where α_p+β_p = a_p, α_p β_p = 1 — Deligne's Sato-Tate).
#
# OK so b_p = a_p, b_{p^2} = a_p^2 - 1 (using Hecke; here a_p = λ_p analytic), b_{p^3} = a_p^3 - 2 a_p, etc.
# In general, b_{p^m} = (α_p^{m+1} - β_p^{m+1})/(α_p - β_p) ... wait that's the Hecke relation for a_{p^m}.
# Actually a_{p^m} is what Hecke gives: a_{p^{m+1}} = a_p a_{p^m} - a_{p^{m-1}} (for level 1).
# And b_{p^m} (von Mangoldt of L) = α_p^m + β_p^m = "Newton's identity" applied to {α,β} with elem syms (a_p, 1).
# So b_p = a_p, b_{p^2} = a_p^2 - 2, b_{p^3} = a_p^3 - 3 a_p, b_{p^4} = a_p^4 - 4 a_p^2 + 2, ...
# (these are different from a_{p^m}!)
#
# So Λ_an(p^m) = log(p) * b_{p^m}. This is what appears in the explicit formula.
#
# The arithmetic side has:
# M_arith[i,j] (prime-power) = - 2 sum_{p^m ≤ X} (log p / p^{m/2}) * b_{p^m} * g_{ij}(m log p)
# where g_{ij}(u) is the Fourier transform of h_{ij}(t) = φ_i(t) φ_j(t).
#
# This is just for context. The full implementation needs the Hermite-Gauss basis and its Fourier transform.
# 
# This is a major implementation. Given remaining time budget, let me commit to delegating to 
# analysis sub-agent now, with explicit instructions including the formulas.

# But — let me first carefully estimate the time and whether the sub-agent can actually do it.
# 105 Q matrix builds + eigenvalue. Each Q is JxJ with J up to 28, so trivial linear algebra.
# Zero finding: ~5 T0 windows, each width ~50 → maybe 50-100 zeros per window → 500 zeros total.
# At dps=80, refined to |Λ|<10^{-70}, each zero takes maybe 5-10 seconds. So 50 minutes for zeros.
# Sub-agent has more cores → can parallelize.

# Let me proceed with sub-agent.
print('Proceeding with sub-agent delegation.')


Proceeding with sub-agent delegation.
